# Tokenizers Fine-Tuning Pipeline (Colab Version)
This notebook consolidates the code to execute the training and evaluation process iteratively across multiple seeds. It is optimized for Google Colab.

## Colab Environment Setup
Run this cell to install the required dependencies.

In [ ]:
!pip install -q transformers datasets scikit-learn
import os
from pathlib import Path


## Configuration

In [ ]:
import json
import os
import random
import numpy as np
import torch

CONFIG = {
  "learning_rate": 2e-5,
  "batch_size": 8,
  "num_epochs": 3,
  "weight_decay": 0.01
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    elif torch.backends.mps.is_available():
        pass # MPS random generator uses torch.manual_seed


## Dataset Handling

In [ ]:
from datasets import load_dataset

# Adjust label list to match all categories in SIB-200 mal_Mlym
LABEL_LIST = [
    "entertainment",
    "geography",
    "health",
    "politics",
    "science/technology",
    "sports",
    "travel",
]

LABEL2ID = {lbl: i for i, lbl in enumerate(LABEL_LIST)}
ID2LABEL = {i: lbl for lbl, i in LABEL2ID.items()}

def load_sib200_malayalam():
    ds = load_dataset("Davlan/sib200", "mal_Mlym")  # has train/validation/test
    
    unique_labels = set(ds["train"]["category"])
    print("Unique labels in train:", unique_labels)
    
    LABEL_FIELD = "category"  # from your screenshot

    def map_labels(example):
        label_str = example[LABEL_FIELD]
        example["label_id"] = LABEL2ID[label_str]
        return example

    ds = ds.map(map_labels)
    return ds

## Tokenization

In [ ]:
# src/tokenization.py
from transformers import AutoTokenizer

MAX_LEN = 128  
MURIL_MODEL_NAME = "google/muril-base-cased"


def morph_shield(text: str) -> str:
    """
    Placeholder for  morphology-aware pre-processing.
    For now, it just returns the text unchanged.
    Later it will insert your suffix-boundary and sandhi rules here.
    """
    # TODO: implement your Malayalam morphological rules
    return text


def load_tokenizer(tokenizer_type: str):
    """
    Always start from MuRIL's own tokenizer so token IDs match
    the pretrained embedding matrix. Baseline = plain MuRIL tokenizer.
    Hybrid = applies morph_shield() to the text before MuRIL tokenization.
    """
    base_tok = AutoTokenizer.from_pretrained(MURIL_MODEL_NAME)

    if tokenizer_type == "baseline":
        # Pure MuRIL tokenizer, no morphological modifications
        return base_tok

    elif tokenizer_type == "hybrid":
        # Wrap MuRIL tokenizer with morphological pre-processing
        class HybridTokenizer:
            def __init__(self, base_tok):
                self.base_tok = base_tok

            def __call__(self, texts, **kwargs):
                # Accept both single string and list of strings
                if isinstance(texts, str):
                    texts = [texts]
                processed = [morph_shield(t) for t in texts]
                return self.base_tok(processed, **kwargs)

            # Some attributes/properties forward to base tokenizer
            @property
            def pad_token_id(self):
                return self.base_tok.pad_token_id

        return HybridTokenizer(base_tok)

    else:
        raise ValueError(f"Unknown tokenizer_type: {tokenizer_type}")


def tokenize_dataset(ds_split, tokenizer):
    """
    Uses the given tokenizer (baseline or hybrid) to produce
    input_ids, attention_mask, and labels for each example.
    """
    def tokenize_batch(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
        )
        enc["labels"] = batch["label_id"]
        return enc

    return ds_split.map(tokenize_batch, batched=True)

## Model Definition

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "google/muril-base-cased"

def load_model(num_labels):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        problem_type="single_label_classification"
    )
    # CRITICAL FIX: Resize the embedding matrix to match custom BPE
    #model.resize_token_embeddings(vocab_size)
    return model



## Training and Evaluation Routine

In [ ]:
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score

def run_training(tokenizer_type, seed):
    print(f"\n>>> Starting run for {tokenizer_type} tokenizer with seed {seed}")
    set_seed(seed)
    cfg = CONFIG

    device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
    print("Using device:", device)

    # 1. Load dataset
    ds = load_sib200_malayalam()

    # 2. Load tokenizer and tokenize splits
    tokenizer = load_tokenizer(tokenizer_type)
    print("Tokenizer loaded.")

    tokenized = {}
    for split in ["train", "validation", "test"]:
        tokenized[split] = tokenize_dataset(ds[split], tokenizer)
        tokenized[split].set_format(
            type="torch",
            columns=["input_ids", "attention_mask", "labels"],
        )

    # 3. Load model
    num_labels = len(ID2LABEL)
    model = load_model(num_labels)
    model.to(device)

    # 4. Data loaders
    train_loader = DataLoader(tokenized["train"], batch_size=cfg["batch_size"], shuffle=True)
    val_loader = DataLoader(tokenized["validation"], batch_size=cfg["batch_size"])
    test_loader = DataLoader(tokenized["test"], batch_size=cfg["batch_size"])

    # 5. Optimizer & scheduler
    optimizer = AdamW(
        model.parameters(),
        lr=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
    )
    num_training_steps = len(train_loader) * cfg["num_epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * num_training_steps),
        num_training_steps=num_training_steps,
    )

    # 6. Training loop
    history = {"val_loss": []}
    for epoch in range(cfg["num_epochs"]):
        print(f"Epoch {epoch+1}/{cfg['num_epochs']}")
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss

            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        # 7. Validation loss
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model(**batch)
                val_losses.append(out.loss.item())
        mean_val_loss = sum(val_losses) / len(val_losses)
        history["val_loss"].append(mean_val_loss)
        print(f"Validation loss: {mean_val_loss:.4f}")

    # 8. Final test evaluation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            labels = batch["labels"].numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits.cpu()
            preds = logits.argmax(dim=-1).numpy()

            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"Test accuracy: {acc:.4f}, macro-F1: {macro_f1:.4f}")

    # 9. Save metrics
    # Local path that works nicely in Colab
    base_path = Path("./")
    run_dir = base_path / "models" / "runs" / f"{tokenizer_type}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    
    with open(run_dir / "metrics.json", "w") as f:
        json.dump(
            {
                "seed": seed,
                "tokenizer": tokenizer_type,
                "accuracy": acc,
                "macro_f1": macro_f1,
                "val_loss": history["val_loss"],
            },
            f,
            indent=2,
        )
    print(f"✅ Run complete. Metrics saved to {run_dir / 'metrics.json'}")


## Execute Full Pipeline (Baseline & Hybrid with Multiple Seeds)

In [ ]:
# Define experiment parameters
seeds = [11, 22, 33, 44, 55]
tokenizer_types = ["baseline", "hybrid"]

# Run experiments
for tok_type in tokenizer_types:
    for s in seeds:
        run_training(tok_type, s)

print("\n🎉 All pipeline runs completed.")
